# Configuração do Servidor Ollama e Modelos Locais

Esta seção inicializa o servidor local Ollama e realiza o download dos modelos necessários para embeddings (`nomic-embed-text`), geração de respostas (`qwen2.5:3b`) e avaliação automática (`qwen2.5:7b` para LLM-as-a-Judge).

In [ ]:
# Installing ollama
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2

In [ ]:
# Running ollama server
import subprocess
import time
import requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up():
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)

print("Server running:", server_is_up())

In [ ]:
# Pull models
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text
!ollama list

# Dependências, Configuração de Caminhos e Provisionamento do Corpus

Instalação dos pacotes necessários (`chromadb`, `openai`, `tiktoken`, `langchain-text-splitters`), configuração de caminhos persistentes (Google Drive ou local) e provisionamento resiliente dos 10 manuais técnicos da Huawei Ascend.

In [ ]:
# Installing dependencies
!pip -q install chromadb openai tiktoken pandas pyarrow tqdm requests langchain-text-splitters

In [ ]:
import os, re, glob, json, math, time, shutil, urllib.request
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
import chromadb

# ============================================================================
# Google Drive & Local Directory Configuration
# ============================================================================
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/RAG_Atlas200DK")
    except Exception:
        BASE_DIR = Path(".")
else:
    BASE_DIR = Path(".")

PROCESSED_MD_DIR = BASE_DIR / "data" / "processed" / "markdown"
OUTPUT_DIR       = BASE_DIR / "data" / "output"
CHROMA_DIR       = str(BASE_DIR / "chroma_db")

for d in [PROCESSED_MD_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL   = "qwen2.5:3b"
JUDGE_MODEL = "qwen2.5:7b"
EMB_MODEL   = "nomic-embed-text"
COLLECTION_NAME = "huawei_ascend_atomic_v3_col"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ============================================================================
# Resilient Corpus Asset Provisioning (10 Markdown Manuals + Chunk Dataset)
# ============================================================================
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ramonvfs/RAG_Atlas200DK/master/"
CHUNK_DATASET_STEM = "huawei_ascend_rag_chunks_atomic"

EXPECTED_DOCS = [
    "ApplicationDevelopmentGuide_MindStudio_.md",
    "ATCToolInstructions.md",
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_.md",
    "EnvironmentDeploymentGuide.md",
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
    "MindStudioInstallationGuideUbuntuX86.md",
    "TBECustomOperatorDevelopmentGuide_CLI_.md",
    "TensorFlowNetworkModelPortingandTrainingGuide.md",
    "UserGuide2020.md",
    "UserGuide2021.md",
]

ALT_ROOTS = [
    Path("."),
    Path("/content/RAG_Atlas200DK"),
    Path("/content"),
    Path("/content/drive/MyDrive/RAG_Atlas200DK"),
]

def _download_file(rel_path: str, dest: Path) -> bool:
    url = GITHUB_RAW_BASE + urllib.request.quote(rel_path)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as out_f:
            out_f.write(resp.read())
        print(f"  [DOWNLOADED] {rel_path}")
        return True
    except Exception as e:
        print(f"  [WARNING] Could not download {rel_path}: {e}")
        return False

def ensure_corpus_assets(md_dir: Path, out_dir: Path) -> tuple:
    # 1. Ensure Markdown manuals exist
    for name in EXPECTED_DOCS:
        target = md_dir / name
        if target.exists():
            continue
        for root in ALT_ROOTS:
            cand = root / "data" / "processed" / "markdown" / name
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {name} from {cand.parent}")
                break
        if not target.exists():
            _download_file(f"data/processed/markdown/{name}", target)

    md_paths = sorted(p for p in md_dir.glob("*.md") if p.is_file())

    # 2. Check for pre-computed chunk dataset (parquet preferred, then json)
    dataset_path = None
    for ext in [".parquet", ".json"]:
        target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
        if target.exists():
            dataset_path = target
            break
        for root in ALT_ROOTS:
            cand = root / "data" / "output" / f"{CHUNK_DATASET_STEM}{ext}"
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {target.name} from {cand.parent}")
                dataset_path = target
                break
        if dataset_path:
            break

    if dataset_path is None:
        for ext in [".parquet", ".json"]:
            target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
            if _download_file(f"data/output/{CHUNK_DATASET_STEM}{ext}", target):
                dataset_path = target
                break

    return md_paths, dataset_path

md_paths, chunk_dataset_path = ensure_corpus_assets(PROCESSED_MD_DIR, OUTPUT_DIR)
print(f"[OK] Located {len(md_paths)} Markdown manuals in {PROCESSED_MD_DIR}")
if chunk_dataset_path:
    print(f"[OK] Found pre-computed chunk dataset: {chunk_dataset_path.name}")

# Estratégias de Chunking: Particionamento Atômico Ciente de Tabelas e Breadcrumbs (Estratégia 3 - Ativa)

Nesta célula, configuramos a **Estratégia 3 (Atomic Table-Aware)** como o particionador ativo do corpus:
- Rebaixa caixas de aviso (`# **NO TE**`, `#### NO TICE`) e pseudocabeçalhos de tabelas/figuras para texto inline em negrito antes do particionamento de seções, preservando a hierarquia estrutural real.
- Mantém tabelas de especificação e blocos de código atômicos e íntegros em chunks individuais.
- Anexa automaticamente até 3 linhas de prosa imediatamente anteriores (título/legenda da tabela) ao corpo da tabela, garantindo o contexto conceitual.
- Injeta breadcrumbs hierárquicos completos de contexto no formato `[Doc: <doc> > <caminho>]`.
- Subdivide seções de prosa que ultrapassam `max_chunk_tokens` através do `RecursiveCharacterTextSplitter`.
- Detecta e preserva marcadores estruturais (`is_table`, `is_code_block`) nos metadados de cada chunk.

A **Estratégia 2 (Markdown Header Splitting)** e a **Estratégia 1 (Recursive Window)** são mantidas implementadas, porém comentadas para referência e benchmarks comparativos.

In [ ]:
import tiktoken
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Tokenizer wrapper using OpenAI cl100k_base
_ENC = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(_ENC.encode(text))

# ============================================================================
# Strategy 3: Atomic Table-Aware & Breadcrumb Hybrid Chunking (ACTIVE)
# ============================================================================
def strategy_atomic_table_aware(
    text: str,
    doc_name: str = "unknown",
    max_chunk_tokens: int = 1024,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Strategy 3: Atomic table-aware chunking with breadcrumb context, callout box
    demotion, and automatic preceding caption attachment for tables.
    """
    # 1. Demote callout boxes and table/figure pseudo-headers before section parsing
    text = re.sub(
        r"^#{1,4}\s*\*?\*?(NO\s*TE|NO\s*TICE|CAUTION|WARNING|DANGER|TIP)\*?\*?\s*$",
        lambda m: f"**{m.group(1).replace(' ', '')}:**",
        text,
        flags=re.MULTILINE | re.IGNORECASE,
    )
    text = re.sub(
        r"^#{1,4}\s*\*?\*?(Table\s+\d+[-\d.]*|Figure\s+\d+[-\d.]*)\*?\*?",
        r"**\1**",
        text,
        flags=re.MULTILINE | re.IGNORECASE,
    )

    lines = text.split("\n")
    sections = []
    current_headers = {}
    current_lines = []
    current_has_table = False

    def _flush_section():
        nonlocal current_lines, current_has_table
        if current_lines:
            path_parts = [current_headers[k] for k in sorted(current_headers.keys())]
            sections.append((
                " > ".join(path_parts) if path_parts else "Root",
                "\n".join(current_lines),
                current_has_table,
            ))
        current_lines = []
        current_has_table = False

    for line in lines:
        header_match = re.match(r"^(#{1,4})\s+(.+)$", line)
        if header_match:
            _flush_section()
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_headers[level] = title
            for k in list(current_headers.keys()):
                if k > level:
                    del current_headers[k]
            current_lines.append(line)
        else:
            current_lines.append(line)
            if line.strip().startswith("|"):
                current_has_table = True
    _flush_section()

    results = []
    last_prose_lines = []

    for header_path, section_text, _ in sections:
        breadcrumb = f"[Doc: {doc_name} > {header_path}]"
        parts = re.split(r"(```[\s\S]*?```|(?:^\|.+\|$\n?){2,})", section_text, flags=re.MULTILINE)

        for part in parts:
            part_stripped = part.strip()
            if not part_stripped:
                continue

            is_table = part_stripped.startswith("|")
            is_code = part_stripped.startswith("```")
            is_atomic = is_table or is_code

            # If this is a table, prepend up to the last 3 lines of preceding prose (table title/caption)
            body_text = part_stripped
            if is_table and last_prose_lines:
                caption_prefix = "\n".join(last_prose_lines[-3:])
                body_text = f"{caption_prefix}\n\n{part_stripped}"

            chunk_text = f"{breadcrumb}\n\n{body_text}"
            tok_count = count_tokens(chunk_text)

            if is_atomic or tok_count <= max_chunk_tokens:
                results.append({
                    "text": chunk_text,
                    "strategy": "atomic_table_aware",
                    "chunk_index": len(results),
                    "approx_tokens": tok_count,
                    "metadata": {
                        "doc_name": doc_name,
                        "header_path": header_path,
                        "is_table": is_table,
                        "is_code_block": is_code,
                    },
                })
            else:
                sub_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=max_chunk_tokens,
                    chunk_overlap=chunk_overlap,
                    length_function=count_tokens,
                )
                for sc in sub_splitter.split_text(part_stripped):
                    sc_with_bc = f"{breadcrumb}\n\n{sc}"
                    results.append({
                        "text": sc_with_bc,
                        "strategy": "atomic_table_aware",
                        "chunk_index": len(results),
                        "approx_tokens": count_tokens(sc_with_bc),
                        "metadata": {
                            "doc_name": doc_name,
                            "header_path": header_path,
                            "is_table": False,
                            "is_code_block": False,
                        },
                    })

            if not is_table:
                non_empty = [ln.strip() for ln in part_stripped.splitlines() if ln.strip()]
                if non_empty:
                    last_prose_lines = non_empty

    return results

print("[OK] Strategy 3 (Atomic Table-Aware) defined and active")

# ============================================================================
# Strategy 2: Markdown Structural / Header Splitting (COMMENTED OUT)
# ============================================================================
# def strategy_markdown_header_split(
#     text: str,
#     doc_name: str = "unknown",
#     max_chunk_tokens: int = 1024,
#     chunk_overlap: int = 64,
# ) -> list[dict]:
#     """
#     Strategy 2: Markdown structural / header splitting.
#
#     Splits along header boundaries (#, ##, ###, ####) and preserves the
#     header hierarchy as metadata. Oversized sections are sub-split with
#     RecursiveCharacterTextSplitter.
#     """
#     # 1. Demote callout boxes and table pseudo-headers before header parsing
#     text = re.sub(
#         r"^#{1,4}\s*\*?\*?(NO\s*TE|NO\s*TICE|CAUTION|WARNING|DANGER|TIP)\*?\*?\s*$",
#         lambda m: f"**{m.group(1).replace(' ', '')}:**",
#         text,
#         flags=re.MULTILINE | re.IGNORECASE,
#     )
#     text = re.sub(
#         r"^#{1,4}\s*\*?\*?(Table\s+\d+[-\d.]*|Figure\s+\d+[-\d.]*)\*?\*?",
#         r"**\1**",
#         text,
#         flags=re.MULTILINE | re.IGNORECASE,
#     )
#
#     headers_to_split_on = [
#         ("#",    "Header_1"),
#         ("##",   "Header_2"),
#         ("###",  "Header_3"),
#         ("####", "Header_4"),
#     ]
#
#     md_splitter = MarkdownHeaderTextSplitter(
#         headers_to_split_on=headers_to_split_on,
#         strip_headers=False,
#     )
#     header_chunks = md_splitter.split_text(text)
#
#     # Sub-split oversized sections
#     sub_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=max_chunk_tokens,
#         chunk_overlap=chunk_overlap,
#         length_function=count_tokens,
#     )
#     results = []
#     for doc in header_chunks:
#         content = doc.page_content
#         metadata = dict(doc.metadata)
#         header_path = " > ".join(
#             v for k, v in sorted(metadata.items())
#             if k.startswith("Header_")
#         ) or "Root"
#
#         breadcrumb = f"[Doc: {doc_name} > {header_path}]"
#         is_table = any(l.strip().startswith("|") for l in content.splitlines())
#         is_code = "```" in content
#
#         if count_tokens(content) > max_chunk_tokens:
#             sub_chunks = sub_splitter.split_text(content)
#             for j, sc in enumerate(sub_chunks):
#                 sc_with_bc = f"{breadcrumb}\n\n{sc}"
#                 results.append({
#                     "text": sc_with_bc,
#                     "strategy": "markdown_header",
#                     "chunk_index": len(results),
#                     "approx_tokens": count_tokens(sc_with_bc),
#                     "metadata": {
#                         "doc_name": doc_name,
#                         "header_path": header_path,
#                         "is_table": any(l.strip().startswith("|") for l in sc.splitlines()),
#                         "is_code_block": "```" in sc,
#                         "sub_chunk": j,
#                     }
#                 })
#         else:
#             c_with_bc = f"{breadcrumb}\n\n{content}"
#             results.append({
#                 "text": c_with_bc,
#                 "strategy": "markdown_header",
#                 "chunk_index": len(results),
#                 "approx_tokens": count_tokens(c_with_bc),
#                 "metadata": {
#                     "doc_name": doc_name,
#                     "header_path": header_path,
#                     "is_table": is_table,
#                     "is_code_block": is_code,
#                 }
#             })
#
#     return results

# ============================================================================
# Strategy 1: Fixed/Recursive Window Splitting (COMMENTED OUT FOR COMPARISON)
# ============================================================================
# def strategy_recursive_split(
#     text: str,
#     doc_name: str = "unknown",
#     chunk_size: int = 1024,
#     chunk_overlap: int = 128,
# ) -> list[dict]:
#     """
#     Strategy 1: Fixed/recursive window splitting (baseline).
#     Splits text based on character separators, ignoring table boundaries.
#     """
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         length_function=count_tokens,
#         separators=["\n\n", "\n", ". ", " ", ""],
#     )
#     chunks = splitter.split_text(text)
#     return [
#         {
#             "text": chunk,
#             "strategy": "recursive",
#             "chunk_index": i,
#             "approx_tokens": count_tokens(chunk),
#             "metadata": {
#                 "doc_name": doc_name,
#                 "header_path": "Root",
#                 "is_table": False,
#                 "is_code_block": False,
#             }
#         }
#         for i, chunk in enumerate(chunks)
#     ]

# Carregamento dos Manuais e Padronização dos Chunks

Esta célula particiona os 10 manuais Markdown utilizando a **Estratégia 3 (Atomic Table-Aware)**. Realiza a limpeza de tags HTML/escapes, correção de OCR em `C0`, exclusão de chunks de sumário (ToC) e preservação atômica das tabelas com legendas associadas.

In [ ]:
LOAD_PRECOMPUTED_CHUNKS = False

_HTML_TAG_RE = re.compile(r"</?(?:span|sup|sub|br|b|i|u|p|div)\b[^>]*>", re.I)
_MD_ESCAPE_RE = re.compile(r"\\([_*$#\[\](){}<>`|~^.!+\-])")

def clean_chunk_text(text: str) -> str:
    text = _HTML_TAG_RE.sub(" ", str(text))
    text = _MD_ESCAPE_RE.sub(r"\1", text)
    # Fix common OCR typo in Ascend manuals where C0 (zero) was extracted as CO (letter O)
    text = re.sub(r"(?<![A-Za-z0-9])CO(?=[,/\s\)\].]|$)", "C0", text)
    return text.replace("**", "").strip()

def is_toc_chunk(text: str) -> bool:
    """True if table has high density of dot leaders or page numbers indicative of Table of Contents."""
    table_lines = [l for l in text.splitlines() if l.strip().startswith("|")]
    if len(table_lines) < 2:
        return False
    dot_count = sum(
        1 for l in table_lines
        if re.search(r"\.{5,}", l) or re.search(r"\|\s*\d+(\.\d+)*\s+.+\|\s*\d+\s*\|", l)
    )
    return dot_count >= max(2, len(table_lines) * 0.4)

rows = []
print(f"[INFO] Computing atomic table-aware chunks (Strategy 3) from {len(md_paths)} Markdown manuals...")
for md_path in md_paths:
    doc_slug = re.sub(r"[^\w\-.]", "_", md_path.stem.strip())
    raw_text = md_path.read_text(encoding="utf-8")
    cleaned_doc_text = clean_chunk_text(raw_text)
    chunks = strategy_atomic_table_aware(cleaned_doc_text, doc_name=md_path.stem)
    for i, c in enumerate(chunks):
        rows.append({
            "id": f"{doc_slug}_{i:04d}",
            "text": clean_chunk_text(c["text"]),
            "doc": md_path.stem,
            "chunk": i,
            "section": clean_chunk_text(c["metadata"]["header_path"]),
            "is_table": c["metadata"]["is_table"],
        })

print(f"[OK] Generated {len(rows):,} raw atomic chunks from {len(md_paths)} documents.")

# Filter out Table of Contents (ToC) chunks
filtered_rows = []
toc_count = 0
for r in rows:
    if is_toc_chunk(r["text"]):
        toc_count += 1
    else:
        filtered_rows.append(r)
rows = filtered_rows
print(f"[INFO] Filtered out {toc_count:,} Table of Contents (ToC) noise chunks. Remaining: {len(rows):,}")

table_chunks = sum(1 for r in rows if r["is_table"])
print(f"[SUMMARY] Total chunks: {len(rows):,} | Specification table chunks: {table_chunks:,} ({table_chunks / len(rows):.1%})")

# Indexação Híbrida: Vetorial (ChromaDB + nomic-embed-text) e Léxica (Okapi BM25)

Criação da coleção vetorial persistente `"huawei_ascend_atomic_v3_col"` usando embeddings densos do modelo `nomic-embed-text` com prefixação assimétrica (`search_document: ` para indexação e `search_query: ` para consultas), integrada a um índice léxico Okapi BM25 em memória com fusão de rankings via Reciprocal Rank Fusion (RRF).

In [ ]:
_LEX_TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[_\-.][A-Za-z0-9]+)*")
_CAMEL_RE = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z0-9]+|[A-Z0-9]+")
_STOPWORDS = set(
    "a an the of to in on for and or is are was were be been being what which who whom how why when where "
    "does do did can could should would will shall may might must this that these those it its with by from "
    "at as into about than then there their them they i my we our you your if not no so such any all each "
    "use used using via per doc section".split()
)

def lexical_tokenize(text: str) -> list[str]:
    out = []
    for m in _LEX_TOKEN_RE.findall(text):
        low = m.lower()
        if low not in _STOPWORDS:
            out.append(low)
        parts = [p for p in re.split(r"[_\-.]", m) if p]
        subs = []
        for p in parts:
            subs.extend(_CAMEL_RE.findall(p) if re.search(r"[a-z][A-Z]", p) else [p])
        if len(subs) > 1:
            out.extend(s.lower() for s in subs if s.lower() not in _STOPWORDS and len(s) > 1)
    return out

class BM25Index:
    """Lightweight Okapi BM25 index over rows for hybrid lexical + dense retrieval."""
    def __init__(self, rows: list[dict], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.rows = rows
        self.id_to_row = {r["id"]: r for r in rows}
        docs_tokens = [lexical_tokenize(r["text"]) for r in rows]
        self.n_docs = len(docs_tokens)
        self.lens = np.array([max(1, len(t)) for t in docs_tokens], dtype=float)
        self.avg_len = float(self.lens.mean()) if self.n_docs else 1.0
        self.postings = defaultdict(list)
        for idx, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((idx, tf))
        self.idf = {
            t: math.log(1.0 + (self.n_docs - len(p) + 0.5) / (len(p) + 0.5))
            for t, p in self.postings.items()
        }

    def search_ids(self, query: str, top_n: int = 15) -> list[str]:
        scores = np.zeros(self.n_docs, dtype=float)
        for term in set(lexical_tokenize(query)):
            if term not in self.postings:
                continue
            idf = self.idf[term]
            for idx, tf in self.postings[term]:
                norm = self.k1 * (1.0 - self.b + self.b * self.lens[idx] / self.avg_len)
                scores[idx] += idf * tf * (self.k1 + 1.0) / (tf + norm)
        top_indices = np.argsort(-scores)[:top_n]
        return [self.rows[i]["id"] for i in top_indices if scores[i] > 0]

BM25_INDEX = BM25Index(rows)
print(f"[OK] BM25 lexical index built over {len(rows):,} chunks ({len(BM25_INDEX.idf):,} unique terms).")

def embed(texts, prefix="search_document: ", model=EMB_MODEL):
    """Obtain dense embeddings via Ollama REST API with asymmetric prefixing."""
    prefixed = [f"{prefix}{t}" for t in texts]
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": prefixed})
    r.raise_for_status()
    return r.json()["embeddings"]

def build_index(rows, name=COLLECTION_NAME, persist_dir=CHROMA_DIR, batch=64, rebuild=False):
    """Build or load a ChromaDB PersistentClient index with batching and progress tracking."""
    store = chromadb.PersistentClient(path=persist_dir)

    if rebuild:
        try:
            store.delete_collection(name)
            print(f"[INFO] Deleted existing collection: {name}")
        except Exception:
            pass

    col = store.get_or_create_collection(
        name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL},
    )

    current_count = col.count()
    if current_count == len(rows):
        print(f"[OK] Collection '{name}' already indexed with {current_count:,} chunks. Skipping re-indexing.")
        return col

    print(f"[INFO] Indexing {len(rows):,} chunks into '{name}' (batch size = {batch})...")
    for i in tqdm(range(0, len(rows), batch), desc="Indexing Chunks", unit="batch"):
        part = rows[i : i + batch]
        col.upsert(
            ids=[r["id"] for r in part],
            documents=[r["text"] for r in part],
            embeddings=embed([r["text"] for r in part], prefix="search_document: "),
            metadatas=[
                {
                    "doc": r["doc"],
                    "chunk": r["chunk"],
                    "section": r["section"],
                    "is_table": r["is_table"],
                }
                for r in part
            ],
        )

    print(f"[OK] Successfully built index! Total items in collection: {col.count():,}")
    return col

def retrieve(col, question, k=5, dense_candidates=15, bm25_candidates=15, rrf_c=60):
    """Hybrid retrieval: fuses ChromaDB dense cosine search with BM25 lexical search via RRF."""
    assert col.metadata.get("embedding_model") == EMB_MODEL, "Embedding model mismatch"
    res = col.query(
        query_embeddings=embed([question], prefix="search_query: "),
        n_results=dense_candidates,
        include=["documents", "metadatas", "distances"],
    )
    dense_hits = {}
    dense_ranking = []
    for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        dense_ranking.append(i)
        dense_hits[i] = {
            "id": i,
            "text": t,
            "doc": m["doc"],
            "section": m["section"],
            "is_table": m.get("is_table", False),
            "distance": float(d),
        }

    bm25_ranking = BM25_INDEX.search_ids(question, top_n=bm25_candidates)

    # Reciprocal Rank Fusion (Dense + BM25)
    fused_scores = defaultdict(float)
    for rank, cid in enumerate(dense_ranking, 1):
        fused_scores[cid] += 1.0 / (rrf_c + rank)
    for rank, cid in enumerate(bm25_ranking, 1):
        fused_scores[cid] += 1.0 / (rrf_c + rank)

    sorted_cids = [cid for cid, _ in sorted(fused_scores.items(), key=lambda x: -x[1])[:k]]
    final_hits = []
    for cid in sorted_cids:
        if cid in dense_hits:
            final_hits.append(dense_hits[cid])
        else:
            r = BM25_INDEX.id_to_row[cid]
            final_hits.append({
                "id": cid,
                "text": r["text"],
                "doc": r["doc"],
                "section": r["section"],
                "is_table": r["is_table"],
                "distance": 0.35,  # fallback distance when surfaced purely by BM25
            })
    return final_hits

def sources_used(hits):
    """Format unique citations as [<doc> · <section>]."""
    seen, out = set(), []
    for h in hits:
        key = f"[{h['doc']} · {h['section']}]"
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

In [ ]:
col = build_index(rows, name=COLLECTION_NAME, batch=64, rebuild=True)

# Motor de Geração RAG e Formatação de Contexto

Configuração do prompt de sistema adaptado ao ecossistema Huawei Ascend e função `ask_rag()` que orquestra a recuperação híbrida (k=5), montagem de contexto estruturado e chamada do modelo `qwen2.5:3b`.

In [ ]:
SYSTEM_ANSWER = (
    "You are a technical assistant specializing in Huawei AI hardware (Ascend 310, Atlas 200 DK) "
    "and the CANN/DDK software ecosystem (ATC, OMG, AscendCL, AIPP, TE/TVM).\n"
    "Answer the user's question strictly using the context provided below. "
    "Be precise with parameter names, compilation flags, data types, and hardware specifications.\n"
    "If the information is not present in the context, objectively declare that you do not have sufficient information."
)

def format_context(hits):
    """Assemble formatted retrieved chunks with document header metadata."""
    parts = []
    for h in hits:
        table_tag = " [Table Chunk]" if h.get("is_table") else ""
        parts.append(f"Document: {h['doc']} (Section: {h['section']}){table_tag}\nContent:\n{h['text']}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, k=5):
    """Full RAG loop: dense retrieval + context formatting + LLM generation."""
    t0 = time.time()
    hits = retrieve(col, question, k=k)
    t_retrieval = time.time() - t0

    context = format_context(hits)
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    t1 = time.time()
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_ANSWER},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
    )
    t_gen = time.time() - t1

    answer = response.choices[0].message.content
    sources = sources_used(hits)

    return {
        "answer": answer,
        "sources": sources,
        "hits": hits,
        "latency_retrieval_s": round(t_retrieval, 3),
        "latency_gen_s": round(t_gen, 3),
        "latency_total_s": round(t_retrieval + t_gen, 3),
    }

# Validação com Suíte Smoke-Test (5 Consultas Técnicas de Domínio)

Execução de 5 consultas técnicas representativas cobrindo os pilares do ecossistema Huawei Ascend para inspecionar a qualidade de recuperação híbrida (k=5), presença de tabelas e respostas do LLM:
1. **Exact Flag / Compiler:** `--input_fp16_nodes` no ATC
2. **Hardware Revisions / Specs:** Diferenças entre IT21DMDA e IT21VDMB
3. **AIPP Preprocessing:** Coeficientes da matriz CSC YUV420SP -> RGB888
4. **Troubleshooting / System Error:** Erro `CheckPartitionSpace partition space check failed`
5. **Tensor Architecture / Memory Layout:** Formato 5D NC1HWC0 e significado de C0

In [ ]:
SMOKE_TEST_QUERIES = [
    {
        "id": "ST1",
        "domain": "Exact Flag / Compiler",
        "target_doc": "ATCToolInstructions.md",
        "query": "What does the --input_fp16_nodes parameter do in the ATC tool?",
    },
    {
        "id": "ST2",
        "domain": "Hardware Revisions / Specs",
        "target_doc": "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
        "query": "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
    },
    {
        "id": "ST3",
        "domain": "AIPP Preprocessing",
        "target_doc": "ATCToolInstructions.md / Atlas200AIAcceleratorModule...",
        "query": "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
    },
    {
        "id": "ST4",
        "domain": "Troubleshooting / System Error",
        "target_doc": "EnvironmentDeploymentGuide.md",
        "query": "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade and how to fix it?",
    },
    {
        "id": "ST5",
        "domain": "Tensor Architecture / Memory Layout",
        "target_doc": "ATCToolInstructions.md / TBECustomOperator...",
        "query": "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16 and INT8?",
    },
]

print(f"Running smoke test on {len(SMOKE_TEST_QUERIES)} Huawei Ascend domain queries...\n")

for item in SMOKE_TEST_QUERIES:
    print("=" * 90)
    print(f"[{item['id']}] Domain: {item['domain']}")
    print(f"Target Doc: {item['target_doc']}")
    print(f"Query: {item['query']}")
    print("-" * 90)

    res = ask_rag(item["query"], k=5)

    print(f"Answer:\n{res['answer']}\n")
    print(f"Latencies: Retrieval={res['latency_retrieval_s']}s | Generation={res['latency_gen_s']}s | Total={res['latency_total_s']}s")
    print("Retrieved Sources:")
    for h in res["hits"]:
        tbl_info = " [TABLE]" if h.get("is_table") else ""
        print(f"  - doc={h['doc']} | dist={h['distance']:.4f}{tbl_info} | section={h['section']}")
    print()

# Técnicas de Pré-Consulta para Otimização do RAG

Implementação das 4 variantes de recuperação pré-consulta para avaliação comparativa:
- **Baseline:** Busca direta pelo texto original da pergunta via recuperação híbrida.
- **Step-Back Prompting:** Abstração da pergunta para recuperar conceitos fundamentais de alto nível.
- **Multi-Query / RAG-Fusion:** Geração de múltiplas reformulações com fusão de rankings via Reciprocal Rank Fusion (RRF).
- **RAG com Memória:** Reescreve perguntas dependentes de contexto histórico em consultas completas e independentes.

In [ ]:
# ============== PARAMETERS CONFIGURATION ==============
K            = 5      # chunks entregues ao gerador
N_RUNS       = 1      # execuções por variante
SEED_BASE    = 42     # seed para reprodutibilidade
GEN_TEMP     = 0.3    # temperatura da resposta final
REWRITE_TEMP = 0.3    # temperatura das chamadas de reescrita
N_VARIANTS   = 5      # reformulações no Multi-Query
RRF_C        = 60     # constante do Reciprocal Rank Fusion
MEMORY_DISTRACTOR = True
FORCE        = True
# =======================================================

class Trace:
    def __init__(self):
        self.llm_calls = 0
        self.embed_calls = 0
        self.queries = []

def llm(messages, tr, temperature, seed):
    tr.llm_calls += 1
    r = client.chat.completions.create(model=GEN_MODEL, messages=messages,
                                       temperature=temperature, seed=seed)
    return r.choices[0].message.content.strip()

def search(col, query, tr, k=K):
    tr.embed_calls += 1
    tr.queries.append(query)
    return retrieve(col, query, k=k)

def rrf(rankings, c=RRF_C):
    """Reciprocal Rank Fusion"""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking, 1):
            fused[cid] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

def dedupe_topk(hits, k=K):
    best = {}
    for idx, h in enumerate(hits):
        # Preserve hybrid RRF rank order while deduplicating
        if h["id"] not in best:
            best[h["id"]] = h
    return list(best.values())[:k]

def clean_lines(text, original, n):
    out = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"')
        if line and line.lower() != original.lower() and len(line) < 300 and line not in out:
            out.append(line)
    return out[:n]

def generate_answer(question, hits, tr, seed):
    messages = [
        {"role": "system", "content": SYSTEM_ANSWER},
        {"role": "user", "content": f"Context:\n{format_context(hits)}\n\nQuestion: {question}"},
    ]
    return llm(messages, tr, GEN_TEMP, seed)

def r_baseline(q, col, tr, seed):
    return q["question"], search(col, q["question"], tr)

### Step-Back Prompting (Abstração de Conceito)

In [ ]:
STEPBACK_SYSTEM = (
    "You reformulate questions for searching Huawei AI accelerator technical documentation (Ascend 310, Atlas 200 DK, CANN, ATC). "
    "Given a specific question, write ONE broader, more general (step-back) question about the underlying concept, architecture, parameter, or technical workflow. "
    "Reply only with the general question.\n\n"
    "Example 1\nQuestion: What does the --input_fp16_nodes flag do during model conversion with ATC?\n"
    "General question: What are the precision control parameters and supported data types of the ATC compiler?\n\n"
    "Example 2\nQuestion: How do I configure matrix_r0c0 to convert YUV420SP to RGB888 in the AIPP configuration file?\n"
    "General question: How does color space conversion (CSC) work in the AIPP preprocessing module?\n\n"
    "Example 3\nQuestion: Why does the CheckPartitionSpace error occur when upgrading the driver on the Atlas 200 DK board?\n"
    "General question: What are the partition requirements and driver upgrade procedures on the Atlas 200 DK?"
)

def r_stepback(q, col, tr, seed):
    out = llm([{"role": "system", "content": STEPBACK_SYSTEM},
               {"role": "user", "content": f"Question: {q['question']}\nGeneral question:"}],
              tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    step_q = lines[0] if lines else q["question"]
    hits = search(col, q["question"], tr) + search(col, step_q, tr)
    return q["question"], dedupe_topk(hits, K)

### Multi-Query / RAG-Fusion (Reformulação com Sinônimos)

In [ ]:
def r_multiquery(q, col, tr, seed):
    system = (
        f"Generate {N_VARIANTS} different technical reformulations of the user's question for searching Ascend and CANN hardware manuals, "
        "using synonyms, alternative phrasing, and technical variations. "
        "Write one per line, without numbering or explanations."
    )
    out = llm([{"role": "system", "content": system}, {"role": "user", "content": q["question"]}],
              tr, REWRITE_TEMP, seed)
    queries = [q["question"]] + clean_lines(out, q["question"], N_VARIANTS)
    rankings, pool = [], {}
    for qq in queries:
        hs = search(col, qq, tr)
        rankings.append([h["id"] for h in hs])
        for h in hs:
            if h["id"] not in pool or h["distance"] < pool[h["id"]]["distance"]:
                pool[h["id"]] = h
    hits = []
    for cid, score in rrf(rankings)[:K]:
        h = dict(pool[cid])
        h["rrf"] = round(score, 5)
        hits.append(h)
    return q["question"], hits

### RAG com Memória (Perguntas de Acompanhamento)

In [ ]:
DISTRACTOR = [
    {
        "role": "user",
        "content": "What is the INT8 computing power of the Ascend 310 Processor?",
    },
    {
        "role": "assistant",
        "content": "The Ascend 310 Processor delivers up to 22 TOPS in INT8 with a typical power consumption of 8W.",
    },
]

MEMORY_SYSTEM = (
    "Given the conversation history and a follow-up question about Huawei Ascend hardware and software, "
    "rewrite the follow-up question so that it is standalone and self-contained. "
    "If it is already standalone, repeat it without changes. Reply only with the rewritten question."
)

def r_memory(q, col, tr, seed):
    history = q.get("history") or (DISTRACTOR if MEMORY_DISTRACTOR else [])
    if not history:
        return q["question"], search(col, q["question"], tr)
    hist = "\n".join(f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in history[-4:])
    out = llm([{"role": "system", "content": MEMORY_SYSTEM},
               {"role": "user", "content": f"History:\n{hist}\n\nFollow-up question: {q['question']}\n\nStandalone question:"}],
               tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    standalone = lines[0] if lines else q["question"]
    return standalone, search(col, standalone, tr)

VARIANTS = {"baseline": r_baseline, "stepback": r_stepback, "multiquery": r_multiquery, "memory": r_memory}

# Suíte de Avaliação Comparativa e LLM-as-a-Judge

Definição das perguntas de teste do domínio Huawei Ascend com gabaritos factuais verificados (`reference`), verificação por palavras-chave (`must_contain`) e avaliação automática imparcial via **LLM-as-a-Judge** (`qwen2.5:7b`).

In [ ]:
os.makedirs("results", exist_ok=True)

def norm(s):
    import unicodedata
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s.replace("*", ""))

def Q(id, tier, question, must_contain, gold_docs, reference, expect_refusal=False, history=None):
    return {
        "id": id,
        "tier": tier,
        "question": question,
        "must_contain": must_contain,
        "gold_docs": gold_docs,
        "reference": reference,
        "expect_refusal": expect_refusal,
        "history": history or [],
    }

ATC_DOC = "ATCToolInstructions"
WP_DOC  = "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08"
ENV_DOC = "EnvironmentDeploymentGuide"
DEV_DOC = "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_"
TBE_DOC = "TBECustomOperatorDevelopmentGuide_CLI_"

QUESTIONS = [
    # ---------------- direct (exact parameters & specs) ----------------
    Q("q1", "direct", "What does the --input_fp16_nodes parameter do in the ATC tool?",
      ["fp16", "input_fp16_nodes"], [ATC_DOC],
      reference="The --input_fp16_nodes parameter specifies the name of the input node(s) with input data of type FP16 (for example, 'node_name1;node_name2'). It is mutually exclusive with --insert_op_conf."),

    Q("q2", "direct", "What are the main hardware differences between IT21DMDA and IT21VDMB board versions of the Atlas 200 DK?",
      ["it21dmda", "it21vdmb", "8", "11"], [WP_DOC],
      reference="According to Table 1-1 of the White Paper: 1) Power input: IT21DMDA accepts 5 V to 28 V, while IT21VDMB accepts 12 V. 2) MIPI-CSI port: IT21DMDA has a 22-pin port (without cables), while IT21VDMB has a 51-pin port (with cables). 3) Number of GPIOs: IT21DMDA has 8 GPIOs, while IT21VDMB has 11 GPIOs. Providing the GPIO difference (8 vs 11) and MIPI-CSI or power difference is sufficient."),

    Q("q3", "direct", "What causes the error 'CheckPartitionSpace partition space check failed' during Atlas 200 DK driver upgrade?",
      ["512", "partition|space|sdcard|sd card"], [ENV_DOC],
      reference="The error occurs because an unallocated space of 512 MB is not reserved on the SD card of the Atlas 200 DK for Driver upgrade (often caused by using an older card creation script). It can be checked with 'fdisk -l' and fixed by resizing the partition (e.g. using GParted on the Ubuntu server to leave 512 MiB of unallocated space)."),

    Q("q4", "direct", "How is the NC1HWC0 5D tensor format structured and what does C0 represent for FP16?",
      ["16", "c0|nc1hwc0"], [ATC_DOC, TBE_DOC, DEV_DOC],
      reference="In the 5D tensor format NC1HWC0, N is batch size, H is height, W is width, C0 is the size of the cube / matrix computing unit in the AI Core, and C1 = (C + C0 - 1) / C0. For FP16, C0 is 16 (and for INT8, C0 is 32)."),

    # ---------------- multi: multiple parameters or tables ----------------
    Q("q5", "multi", "Which CSC matrix coefficients are used in AIPP to convert YUV420SP_U8 to RGB888_U8 for JPEG images?",
      ["matrix_r0c0", "256|298"], [ATC_DOC, DEV_DOC],
      reference="In Table 4-1 (YUV420SP_U8 to RGB / RGB888_U8), for JPEG images the CSC matrix coefficients are matrix_r0c0: 256, matrix_r0c1: 0, matrix_r0c2: 359, matrix_r1c0: 256, matrix_r1c1: -88, matrix_r1c2: -183, matrix_r2c0: 256, matrix_r2c1: 454, matrix_r2c2: 0 (with input_bias 0, 128, 128). For BT.601 Narrow, the coefficients are 298, 0, 409, 298, -100, -208, 298, 516, 0. Citing either the JPEG column (256...) or the BT.601 column (298...) from the YUV420SP_U8 to RGB table is acceptable."),

    Q("q6", "multi", "What is the typical power consumption and power input voltage range of the Atlas 200 DK?",
      ["12", "20|36|28"], [WP_DOC],
      reference="According to the White Paper (Sections 3.1 and 4.4): Typical power consumption is 20 W, and the external power supply adapter must be >= 36 W (12 V 3 A configured by default). The power input voltage is 12 V if the PCB is IT21VDMB, and 5 V to 28 V if the PCB is IT21DMDA. Mentioning the 12V / 5V-28V input voltages and either the 20W typical power or >=36W power supply requirement is correct."),

    # ---------------- paraphrase: vocabulary gap ----------------
    Q("q7", "paraphrase", "How do I configure channel mean subtraction and variance normalization in the AIPP configuration file?",
      ["mean_chn|var_reci_chn|min_chn"], [ATC_DOC, DEV_DOC],
      reference="In the AIPP configuration file (aipp_op), channel normalization is configured using mean_chn_0..3 (mean value of each channel), min_chn_0..3 (minimum value of each channel), and var_reci_chn_0..3 (variance / reciprocal factor of each channel), following the rule: pixel_out = (pixel_in - mean_chn - min_chn) * var_reci_chn."),

    Q("q8", "paraphrase", "What should I do if the Atlas board cannot establish connection to the Ubuntu server via USB?",
      ["usb", "ip|nic|cable|led|sd"], [ENV_DOC, "UserGuide2021", "UserGuide2020"],
      reference="According to the troubleshooting guide (What Do I Do If the Atlas 200 DK Cannot Connect to the Ubuntu Server?): 1) Check SD card preparation logs and ensure LED1/LED2 indicators are normal. 2) Check the USB cable and verify if the virtual USB NIC exists and its IP address (e.g. 192.168.1.x) is configured on the Ubuntu server. 3) Alternatively, try connecting via network cable (NIC mode) or connect via a serial cable (UART) using a debugging tool (e.g. IPOP / PL2303 driver) to inspect boot logs."),

    # ---------------- followup: depends on conversation history ----------------
    Q("q9", "followup", "And what is its value for INT8 data types?",
      ["32"], [ATC_DOC, TBE_DOC, DEV_DOC],
      reference="For INT8 data types in the NC1HWC0 5D format, C0 is 32.",
      history=[{"role": "user", "content": "What is the value of C0 for FP16 in the NC1HWC0 5D tensor format?"},
               {"role": "assistant", "content": "In the NC1HWC0 5D tensor format, C0 represents the size of the cube unit in the AI Core and equals 16 for FP16."}]),

    # ---------------- none: out of domain refusal ----------------
    Q("q10", "none", "What is the GPU clock speed of Nvidia RTX 4090?",
      [], [],
      reference="Out of domain question. The answer must state that the information is not provided in the context.",
      expect_refusal=True),
]

QUESTIONS.sort(key=lambda q: int(q["id"][1:]))

with open("results/questions.json", "w", encoding="utf-8") as f:
    json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)

print(f"[OK] Benchmark suite: {len(QUESTIONS)} questions configured with verified Huawei manual references.")

In [ ]:
REFUSAL_RE = re.compile(
    r"do not have (enough|sufficient) information|don't have (enough|sufficient) information"
    r"|insufficient information|not enough information|cannot (answer|determine|find)"
    r"|does not (contain|mention|provide|specify)|not (mentioned|provided|specified|found|available) in the (provided )?context"
    r"|no information (available|provided|found)"
    r"|nao (possui|possuo|tenho|encontr|ha informac|consta)|informacoes suficientes",
    re.I,
)

def has_kw(text_norm, kw):
    return any(norm(alt).strip() in text_norm for alt in kw.split("|"))

def grade(answer, q):
    """
    Evaluate answer using LLM-as-a-judge (qwen2.5:7b) comparing candidate answer
    against factual ground-truth reference, with deterministic refusal handling.
    """
    a = norm(answer)
    refused = bool(REFUSAL_RE.search(a))
    kw_ok = refused if q["expect_refusal"] else all(has_kw(a, kw) for kw in q["must_contain"])

    # 1. Deterministic handling for out-of-domain refusal questions (e.g. q10)
    if q["expect_refusal"]:
        if refused:
            grade.last = {"reason": "Correctly refused out-of-domain question based on context."}
            return True, True
        else:
            grade.last = {"reason": "Failed to refuse out-of-domain question."}
            return False, False

    # 2. If it is an answerable question but the RAG model refused to answer
    if refused:
        grade.last = {"reason": "Model stated insufficient information instead of answering."}
        return False, True

    reference = q.get("reference") or "; ".join(q["must_contain"])
    hist = "\n".join(f"{t['role']}: {t['content']}" for t in q.get("history", [])) or "(no history)"

    judge_prompt = (
        f"Conversation history:\n{hist}\n\n"
        f"Question: {q['question']}\n\n"
        f"Ground truth reference:\n{reference}\n\n"
        f"Candidate answer to evaluate:\n{answer}\n\n"
        "Evaluate whether the candidate answer is CORRECT based on the ground truth reference:\n"
        "1. It must accurately answer the question and align with the key facts in the reference.\n"
        "2. Do NOT penalize the candidate answer for including additional valid technical steps or details from the manual that are not in the short reference summary.\n"
        "3. Mark correct=false ONLY if the answer misses the core technical answer or directly contradicts the reference numbers/facts.\n"
        'Reply ONLY in JSON format: {"correct": true or false, "reason": "a short sentence"}'
    )

    try:
        r = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": "You are a fair, technically accurate evaluator of RAG system answers."},
                {"role": "user", "content": judge_prompt},
            ],
            temperature=0, seed=0, response_format={"type": "json_object"}
        )
        v = json.loads(r.choices[0].message.content)
        val = v.get("correct") if "correct" in v else v.get("correto")
        judge_ok = val is True or str(val).strip().lower() in ("true", "1")
        reason = str(v.get("reason") or v.get("motivo") or "")[:180]
    except Exception as e:
        judge_ok, reason = kw_ok, f"judge fallback ({type(e).__name__}): keywords used"

    grade.last = {"reason": reason}
    return judge_ok, refused

def run_variant(label, fn, questions, run, col):
    path = f"results/{label}_r{run}.csv"
    if os.path.exists(path) and not FORCE:
        return pd.read_csv(path)
    rows = []
    for qi, q in enumerate(questions):
        tr, seed = Trace(), SEED_BASE + 1000 * run + qi
        t0 = time.time()
        q_used, hits = fn(q, col, tr, seed)
        answer = generate_answer(q_used, hits, tr, seed)
        latency = time.time() - t0

        correct, refused = grade(answer, q)
        judge_reason = grade.last.get("reason", "")

        docs = {h["doc"] for h in hits}
        recall = (sum(d in docs for d in q["gold_docs"]) / len(q["gold_docs"])) if q["gold_docs"] else np.nan
        ctx = norm(format_context(hits))
        ctx_keys = all(has_kw(ctx, kw) for kw in q["must_contain"]) if q["must_contain"] else np.nan

        rows.append({
            "judge_reason": judge_reason,
            "label": label, "run": run, "id": q["id"], "tier": q["tier"], "question": q["question"],
            "question_used": q_used, "queries": json.dumps(tr.queries, ensure_ascii=False),
            "retrieved": " | ".join(f"{h['id']}({h['distance']:.3f})" for h in hits),
            "gold_docs": ";".join(q["gold_docs"]), "gold_recall": recall, "ctx_has_keys": ctx_keys,
            "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": tr.llm_calls, "embed_calls": tr.embed_calls, "latency_s": round(latency, 2),
        })
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

In [ ]:
# Evaluates variants on questions.json (optional quick run or batch evaluation)
RUN_FULL_EVALUATION = False  # Set to True when ready to execute all variants in Google Colab

if RUN_FULL_EVALUATION:
    with open("results/questions.json", encoding="utf-8") as f:
        questions = json.load(f)

    # Warmup call
    llm([{"role": "user", "content": "ok"}], Trace(), 0.0, 0)
    embed(["ok"], prefix="search_query: ")

    for label, fn in VARIANTS.items():
        for run in range(1, N_RUNS + 1):
            t0 = time.time()
            df = run_variant(label, fn, questions, run, col)
            print(f"{label:<11} r{run}  acc={df['correct'].mean():.0%}  llm_calls/pergunta={df['llm_calls'].mean():.1f}  ({time.time()-t0:.0f}s)")
        pd.concat([pd.read_csv(f"results/{label}_r{r}.csv") for r in range(1, N_RUNS + 1)]) \
          .to_csv(f"results/results_{label}.csv", index=False)

    try:
        ollama_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json().get("version")
    except Exception:
        ollama_version = None
    with open("results/run_config.json", "w", encoding="utf-8") as f:
        json.dump({"gen_model": GEN_MODEL, "judge_model": JUDGE_MODEL, "emb_model": EMB_MODEL,
                   "ollama_version": ollama_version, "k": K, "n_runs": N_RUNS, "seed_base": SEED_BASE,
                   "gen_temp": GEN_TEMP, "rewrite_temp": REWRITE_TEMP, "n_variants": N_VARIANTS, "rrf_c": RRF_C,
                   "memory_distractor": MEMORY_DISTRACTOR, "n_chunks": len(rows), "n_questions": len(questions),
                   "date": time.strftime("%Y-%m-%d %H:%M")}, f, indent=2, ensure_ascii=False)
    print("\n[OK] Comparative results exported to results/")
else:
    print("[INFO] Set RUN_FULL_EVALUATION = True to execute the complete variant benchmark with LLM-as-a-judge.")

# Comparativo de Desempenho e Motivo da Avaliação do Juiz

Exibição detalhada das respostas de cada variante com o veredito (`[OK]` / `[FAIL]`) e a justificativa emitida pelo modelo juiz (`qwen2.5:7b`), acompanhada do resumo de acurácia comparativa.

In [ ]:
from IPython.display import display
pd.set_option("display.max_colwidth", 300)

summary_rows = []
for label in VARIANTS:
    csv_path = f"results/results_{label}.csv"
    if not os.path.exists(csv_path):
        continue
    d = pd.read_csv(csv_path)

    t = d[["id", "question", "answer"]].copy()
    t["result"] = d["correct"].map({True: "[OK]", False: "[FAIL]"})
    t["judge_reason"] = d["judge_reason"]

    correct_cnt, total_cnt = int(d["correct"].sum()), len(d)
    summary_rows.append({
        "variant": label,
        "correct": correct_cnt,
        "total": total_cnt,
        "accuracy": f"{correct_cnt / total_cnt:.0%}",
    })

    print(f"\n===== Variant: {label} =====")
    display(t.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    print(f"Accuracy: {correct_cnt}/{total_cnt} = {correct_cnt / total_cnt:.0%}")

if summary_rows:
    print("\n===== SUMMARY COMPARISON =====")
    display(pd.DataFrame(summary_rows).style.hide(axis="index"))